In [ ]:
if "BrentScheme" not in locals():
  !pip install "git+https://github.com/perryGabriel/brent-scheme.git"
  from brentscheme import *
import numpy as np
import torch
from tqdm import trange
import pandas as pd
import time
import matplotlib.pyplot as plt
from google.colab import files

In [ ]:
scheme = BrentScheme(n=3, d=2, m=4, p=8, verbose=1)
factory = SchemaFactory()
printer = SchemeDisplay()
manipulator = SchemeManipulator()
stepper = Stepper()
trainer = Trainer()

A scheme for (3 x 2) @ (2 x 4) using 8 products: complexity is n^1.963


In [ ]:
# @title File Save/Read Functions

# printer.test(scheme, verbose=0)
# round = lambda x,y:x # needed if dump_to_file throws an error - remember to use number= to avoid a long file name

# Save the scheme to Colab
# manipulator.set_norm(scheme, 2)
# score = printer.dump_to_file(scheme, number = -3.145)
# print(f"Score: {score}")

# Download the file from Colab to your computer. Remember to specify which scheme you want!
# download_file(n=scheme.n, d=scheme.d, m=scheme.m, p=scheme.p, number=score, scheme_or_diagram='scheme')

# Read into locals() a scheme from Colab's files. Remember to specify which scheme you want!
# factory.set_scheme(scheme, 'random', n=3, d=3, m=3, p=22)
# factory.read_from_files(scheme, n=scheme.n, d=scheme.d, m=scheme.m, p=scheme.p, number=-3.130, verbose=2)

# Delete a scheme from Colab. Remember to specify which scheme you want!
# delete_file(n=scheme.n, d=scheme.d, m=scheme.m, p=scheme.p, number=score, scheme_or_diagram='scheme')

In [ ]:
#@title The above degrees of freedom are equal to the null space of the Jacobian
# can use this to find dimension of solution manifold (take null space of Jacobian) DID NOT WORK - SINGULAR VALUES ARE MIXED UP (Naive scheme had a null space...)
# can also use it to find step direction - if detlas-scheme == err = Jacobian @ params, params = J_inv @ err
# i.e. small perturabtions in these directions do not affect the scheme.


# factory.set_scheme(scheme, 'naive', n=3, d=3, m=3)
# factory.set_scheme(scheme, 'laderman', n=3, d=3, m=3, p=23)
# factory.set_scheme(scheme, 'random', n=3, d=3, m=3, p=22)
factory.read_from_files(scheme, n=scheme.n, d=scheme.d, m=scheme.m, p=scheme.p, number=-3.145, verbose=0)# factory.set_scheme(scheme, 'naive', n=2, d=2, m=2, p=8)
# factory.set_scheme(scheme, 'random', n=2, d=2, m=2, p=8)
stepper.epoch_pseudoinverse(scheme, batch_size=800)
manipulator.set_norm(scheme, 1)
printer.print(scheme, verbose=1)

num_iters_each_round = 200
step_size = 1e-7 # Do binary search. Differnet zoom factors are deceiving - not a smooth surface at any magnification level.
scores = []
delta_alpha, delta_beta, delta_gamma = stepper.get_abs_gradient_direction(scheme)
delta_alpha, delta_beta, delta_gamma = delta_alpha * step_size, delta_beta * step_size, delta_gamma * step_size

backup_iters = num_iters_each_round//2
scheme.alpha_pnd -= backup_iters * delta_alpha
scheme.beta__pdm -= backup_iters * delta_beta
scheme.gamma_nmp -= backup_iters * delta_gamma

for i in range(num_iters_each_round): # 228, 129,
  scheme.alpha_pnd += delta_alpha
  scheme.beta__pdm += delta_beta
  scheme.gamma_nmp += delta_gamma
  scores.append(printer.test(scheme, verbose=0))

backup_iters = len(scores)-1 - torch.argmin(torch.tensor(scores))
scheme.alpha_pnd -= backup_iters * delta_alpha
scheme.beta__pdm -= backup_iters * delta_beta
scheme.gamma_nmp -= backup_iters * delta_gamma

plt.axvline(x=torch.argmin(torch.tensor(scores)), color='red', linestyle=':')
plt.axhline(y=printer.test(scheme, verbose=0), color='red', linestyle=':')
plt.title(f"Min Acheived at {torch.argmin(torch.tensor(scores))} Iterations: {torch.min(torch.tensor(scores))}")
plt.plot(scores)
plt.show()
printer.print(scheme, verbose=2)

In [ ]:
#@title Directional Search
def find_min_one(scheme, delta_alpha, delta_beta, delta_gamma, step_size=1e-2, max_num_iters=500, verbose=0):
  # precomputing single steps saves multiplication time, but increasing the step size makes for fewer steps
  # delta_alpha, delta_beta, delta_gamma = delta_alpha*step_size, delta_beta*step_size, delta_gamma*step_size

  scores = [printer.test(scheme, verbose=0)]
  for iter in range(max_num_iters): ### FIXME: Change to a binary search (higher than start = other side, binary search in middle)
    scheme.alpha_pnd += delta_alpha *step_size
    scheme.beta__pdm += delta_beta *step_size
    scheme.gamma_nmp += delta_gamma *step_size
    scores.append(printer.test(scheme, verbose=0))
    if scores[-1] > scores[-2]:
      del scores[-1]
      break
    step_size *= 1.05

  # back up one
  scheme.alpha_pnd -= delta_alpha *step_size
  scheme.beta__pdm -= delta_beta *step_size
  scheme.gamma_nmp -= delta_gamma *step_size

  if verbose > 0: return scheme, scores
  else: return scheme

In [ ]:
 # Do binary search. Differnet zoom factors are deceiving - not a smooth surface at any magnification level.
 # Add higher order terms - get differences between shorter steps and try to anticipate the curve

In [ ]:
#@title Get Continued Pseudoinverse Gradient
def pseudoinverse_gradient(scheme, batch_size=1000):
  prev = scheme.clone()
  stepper.epoch_pseudoinverse(scheme, batch_size=batch_size)
  # delta is the step from the old to the new
  delta_alpha, delta_beta, delta_gamma = scheme.alpha_pnd - prev.alpha_pnd, scheme.beta__pdm - prev.beta__pdm, scheme.gamma_nmp - prev.gamma_nmp

  return scheme, delta_alpha, delta_beta, delta_gamma

In [ ]:
#@title Get Gradient Using Pytorch
def torch_gradient(scheme, lr=1e-3):
  scheme.alpha_pnd.requires_grad = True
  scheme.beta__pdm.requires_grad = True
  scheme.gamma_nmp.requires_grad = True

  optimizer = optim.Adam([scheme.alpha_pnd, scheme.beta__pdm, scheme.gamma_nmp], lr=lr)

  loss = torch.nn.MSELoss()
  optimizer.zero_grad()
  output = torch.einsum("cCi,iaA,ibB->cCaAbB", scheme.gamma_nmp, scheme.alpha_pnd, scheme.beta__pdm)
  target = scheme.TRIPLE_DELTA_nmnddm
  cost = loss(output, target)
  cost.backward()
  delta_alpha, delta_beta, delta_gamma = -scheme.alpha_pnd.grad, -scheme.beta__pdm.grad, -scheme.gamma_nmp.grad

  scheme.alpha_pnd = scheme.alpha_pnd.cpu().detach().type(torch.float64)
  scheme.beta__pdm = scheme.beta__pdm.cpu().detach().type(torch.float64)
  scheme.gamma_nmp = scheme.gamma_nmp.cpu().detach().type(torch.float64)

  return delta_alpha, delta_beta, delta_gamma

In [ ]:
#@title Instead of the Jacobian, try using the steps made by pseudoinverse (continue in direction of that motion)
# log the changes on each iteration; perhaps they are moving along a curve in the parameter space, and what if we could know what that curve is? Or add momentum to the pseudoinverse?
# measure the normalized dot product between sucessive steps to track if they are in a relatively straight line. Do this with differnce of differences for second derivative, etc.

# factory.set_scheme(scheme, 'random', n=3, d=3, m=3, p=22)
# stepper.epoch_pseudoinverse(scheme, batch_size=3000)
# factory.read_from_files(scheme, n=scheme.n, d=scheme.d, m=scheme.m, p=scheme.p, number=-3.145, verbose=0)# factory.set_scheme(scheme, 'naive', n=2, d=2, m=2, p=8)
# factory.set_scheme(scheme, 'random', n=2, d=2, m=2, p=8)
manipulator.set_norm(scheme, 2)
printer.print(scheme, verbose=1)

num_rounds = 100
step_size = 3e-2
scores = [printer.test(scheme, verbose=0)]

for round in trange(1, num_rounds+1):
  grad = (None,None,None)

  # grad = stepper.get_abs_gradient_direction(scheme) # does not work as well as hoped

  scheme, *grad = pseudoinverse_gradient(scheme, batch_size=300) # works very well
  # pseudo_grad = torch.cat((grad[0].flatten(),grad[1].flatten(),grad[2].flatten()))
  # pseudo_grad_norm = torch.norm(pseudo_grad)
  # display(pseudo_grad_norm)

  # grad = torch_gradient(scheme)
  # torch_grad = torch.cat((grad[0].flatten(),grad[1].flatten(),grad[2].flatten()))
  # torch_grad_norm = torch.norm(torch_grad)
  # display(torch_grad_norm)

  # angle = np.arccos(torch.dot(pseudo_grad, torch_grad) / (pseudo_grad_norm * torch_grad_norm))
  # display(angle)
  # print()

  scores.append(printer.test(scheme, verbose=0))
  plt.axvline(x=len(scores)-1, color='red', linestyle=':')
  scheme, _scores = find_min_one(scheme, *grad, step_size=step_size, max_num_iters=5000, verbose=1)
  scores.extend(_scores)

plt.plot(scores)
plt.show()
printer.print(scheme, verbose=2)